In [ ]:
!pip install -q langchain langchain-core chromadb groq gradio langchain_community langchain-groq sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.

In [ ]:
import os
from groq import Groq
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')

if groq_api_key:
    print(f"Loaded Groq API Key (first 5 chars): {groq_api_key[:5]}*****")
else:
    print("GROQ_API_KEY not found in Colab secrets.")

try:
    client = Groq(api_key=groq_api_key)
    print("Groq client initialized successfully.")
except Exception as e:
    print(f"Error initializing Groq client. Make sure your API key is set correctly. Error: {e}")
    client = None

Loaded Groq API Key (first 5 chars): gsk_0*****
Groq client initialized successfully.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader
)
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

CHROMA_DIR = "chroma_store"

# Initialize models
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Initialize ChromaDB
vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embedding_model
)

# Initialize LLM
if groq_api_key:
    llm = ChatGroq(
        groq_api_key=groq_api_key,
        model_name="llama-3.3-70b-versatile",
        temperature=0.2
    )
else:
    llm = None
# --- Step 3: Logic Functions ---

def index_files(files):
    if not files:
        return "No files uploaded."

    docs = []
    for file in files:
        path = file.name
        ext = path.lower().split(".")[-1]

        try:
            if ext == "pdf":
                loader = PyPDFLoader(path)
            elif ext == "txt":
                loader = TextLoader(path)
            elif ext == "csv":
                loader = CSVLoader(file_path=path)
            else:
                continue
            docs.extend(loader.load())
        except Exception as e:
            print(f"Error loading {path}: {e}")

    if not docs:
        return "No valid text found in uploaded files."

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)
    vectorstore.add_documents(chunks)

    return f"✅ Indexed {len(chunks)} chunks from {len(files)} files"

def ask_question(query):
    if not llm:
        return "LLM not initialized. Check your Groq API key."

    # Simple check to see if we have data
    if not vectorstore._collection.count():
        return "The vector database is empty. Please upload and index files first."

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    context = "\n\n".join(d.page_content for d in docs)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer the user's question using ONLY the provided context. If the answer is not in the context, say you don't know."),
        ("human", "Context:\n{context}\n\nQuestion:\n{question}")
    ])

    messages = prompt.format_messages(
        context=context,
        question=query
    )

    response = llm.invoke(messages)
    return response.content


In [ ]:
import gradio as gr
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔍 PragyanAI - RAG Chatbot (Groq + ChromaDB)")
    gr.Markdown("Upload documents to build a knowledge base, then ask questions about them.")

    with gr.Row():
        with gr.Column(scale=1):
            file_upload = gr.File(
                label="Step 1: Upload Documents",
                file_types=[".pdf", ".txt", ".csv"],
                file_count='multiple'
            )
            index_btn = gr.Button("📥 Index Files", variant="primary")
            index_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            question = gr.Textbox(label="Step 2: Ask a Question", placeholder="What is the main topic of the PDF?")
            ask_btn = gr.Button("🤖 Get Answer", variant="secondary")
            answer = gr.Textbox(label="Response", lines=10)

    # Event handlers
    index_btn.click(index_files, inputs=[file_upload], outputs=[index_status])
    ask_btn.click(ask_question, inputs=[question], outputs=[answer])
    question.submit(ask_question, inputs=[question], outputs=[answer])

# Launch with share=True for Colab link accessibility
demo.launch(share=True, debug=True)

/tmp/ipython-input-3612811975.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1c51b422ef116a6933.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1c51b422ef116a6933.gradio.live


In [ ]:
!pip install llama-index groq llama-index-llms-groq

In [ ]:
# Import OS module (useful for environment variables and system operations)
import os

# Import the Groq LLM wrapper from LlamaIndex
# This allows LlamaIndex to use Groq-hosted large language models
from llama_index.llms.groq import Groq

# Import global settings object from LlamaIndex
# Used to configure the default LLM across the entire application
from llama_index.core import Settings

# Import Colab utility to safely access stored secrets (API keys)
from google.colab import userdata


# -------------------------------------------------
# Retrieve the Groq API key securely
# -------------------------------------------------
# The API key should be stored in Colab Secrets as "GROQ_API_KEY"
# This avoids hardcoding sensitive credentials in your notebook
groq_api_key = userdata.get('GROQ_API_KEY')


# -------------------------------------------------
# Initialize the Groq Large Language Model
# -------------------------------------------------
# llama-3.3-70b-versatile:
# - Very large model (70B parameters)
# - Excellent at reasoning, math, and complex instructions
# - Runs extremely fast on Groq LPUs
#
# The Groq class handles:
# - Authentication
# - API communication
# - Prompt execution
llm = Groq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key
)


# -------------------------------------------------
# Set Groq LLM as the default model for LlamaIndex
# -------------------------------------------------
# Any LlamaIndex component (query engines, agents, tools)
# will now automatically use this LLM unless overridden
Settings.llm = llm


# -------------------------------------------------
# Confirmation message
# -------------------------------------------------
print("LlamaIndex successfully powered by Groq LPU.")